# Poetry through the eyes of a machine — follow-up
Run all cells in Colab. Embedded private corpus and original challenge prompts. All new alternatives are AI-drafted, not independently author-validated. This is exploratory, not a controlled proof of comprehension. Full-poem comparisons change both context and length; overlap categories are intended, not guaranteed. Scores are compared within each model.

In [ ]:
%pip -q install sentence-transformers scikit-learn pandas matplotlib

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, shutil, re, json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files
out=Path('poetry_followup'); out.mkdir(exist_ok=True)
poems=pd.DataFrame([{'poem_id': 'P01', 'title': 'The Painter', 'poem_text': 'there was a painter\nwho once said\nthat reality is the least honest thing in the world.\n\neveryone laughed at him for that.\n\nbecause reality is measurable, isn’t it?\ntouchable. visible.\nit exists in definite shapes\nand names that stay still long enough\nfor language to hold them.\n\nbut he kept thinking of all the things\nthat had changed him completely\nwithout ever becoming visible at all.\n\nhow do you paint the feeling\nof returning home\nand realising you no longer belong to it\nin quite the same way?\n\nhow do you paint the exact moment\nsomeone stops being a person you know\nand quietly becomes part\nof the architecture of your inner world?\n\nhow do you paint longing\nbefore it learns its own name?\n\nhe spent years trying.\n\nnot painting people…\npeople were easy.\nfaces are only geometry.\nlight resting temporarily on bone.\n\nhe wanted to paint the invisible movements beneath them.\n\nthe unbearable tenderness\nof someone remembering the smallest thing about you.\n\nthe strange ache\nof watching time pass through another person’s face.\n\nthe way grief does not arrive screaming,\nbut instead slips silently into ordinary objects;\na half-read book,\nan untouched cup of coffee,\na chair angled slightly toward the door\nas if still waiting.\n\nhe painted in impossible colors.\n\nnot red,\nbut the color of almost becoming.\n\nnot blue,\nbut the color of memories\nthat feel more alive\nthan the moments themselves ever did.\n\npeople looked at his canvases and said,\n“this doesn’t look real.”\n\nand he always wanted to ask them…\n\nwas the trembling in your chest real\nthe first time you realized\nanother human being\nhad somehow altered the gravity inside you?\n\nis music unreal\nbecause you cannot hold it in your hands?\n\nis a dream less true\nsimply because morning survives it?\n\nhe believed truth lived elsewhere.\n\nnot in objects themselves,\nbut in the silent spaces surrounding them.\n\nin pauses during conversations.\nin the unbearable intimacy of being understood accidentally.\nin the silence after laughter fades,\nwhen you suddenly become aware\nof how temporary every beautiful thing is.\n\nthat was what he painted.\n\nnot reality itself,\nbut the wound reality leaves behind\nwhen it touches the soul deeply enough.\n\nand one night,\nafter years of failure,\nhe painted something that frightened him.\n\nit was not a person.\nnot a landscape.\nnot anything recognizable.\njust light\nfalling through a half open door\ninto an empty room.\n\nbut somehow,\ninside that room\nlived every goodbye he had never recovered from,\nevery version of himself\nhe had abandoned to survive,\nevery fleeting beautiful thing\nhe had loved too deeply\nto keep.\n\nhe stared at the canvas for hours afterward.\n\nbecause for the first time in years,\nhe had not painted what something looked like.\n\nhe had painted\nwhat it meant to lose it.\n\nand perhaps that is the closest art ever comes\nto telling the truth…\n\nnot by reproducing the world faithfully,\nbut by revealing the invisible devastations\nrequired to survive it.\n\nbecause the truth does not live\nin reality itself.\n\nit lives in the place\nwhere reality ends\nand feeling begins.'}, {'poem_id': 'P02', 'title': 'These Mountains', 'poem_text': "these mountains that you are holding,\nyou were only supposed to climb.\n\nthey were never meant for your shoulders,\nthey were meant for your steps.\n\nfor the shaky knees,\nthe lungs that burn,\nthe soul that feels lighter each time.\n\ni thought maybe the ache was proof of living—\nbut no, love,\nyou weren't born to collapse under the sky.\n\nyou were born to touch it.\n\nso set it down.\nbreathe.\njust climb."}, {'poem_id': 'P03', 'title': 'Forward Is Forward', 'poem_text': 'forward is forward,\neven when it feels like nothing changed today.\nyou are still moving,\nstill weaving little threads of tomorrow\nwith every breath you take.\n\nthe river doesn’t rush all the time—\nsometimes it drifts,\nsometimes it rests,\nbut it always flows.\n\nso do you.\n\nyou don’t need to hurry.\nyou don’t need to prove.\nyou are enough in your becoming,\nenough in your slow and steady light.\n\nrest in that truth:\nyou are on your way,\nand that is more than enough.'}, {'poem_id': 'P04', 'title': 'Broken Glass', 'poem_text': 'she didn’t just pick up the pieces,\nshe bled on them,\nshe carved her pain into them,\nand built a mosaic the world had never seen.\n\nthe world called her broken,\nbut they didn’t know—\nbroken glass reflects more light\nthan something whole ever could.\n\nshe was no longer a girl piecing herself together,\nshe was the fire that rose from her own ashes.\nand when she walked,\nthe air shifted—\nfor she carried a new truth:\nbroken isn’t the end,\nbroken is the beginning.'}, {'poem_id': 'P05', 'title': 'rooms that remember truth', 'poem_text': '.\nthere’s a kind of magic in walking slow,\nthrough halls where the light feels older than the walls.\nit lands in soft pools\non leather chairs,\ndusty book edges,\nhalf-sipped tea.\neverything still.\neverything listening.\n.\nthe paintings don’t speak—\nbut they watch.\nand in that watching,\nyou realize:\nsome things don’t need words\nto be understood.\n.\nthere’s a book that’s been waiting for me\nfor years.\nnot because it holds answers—\nbut because it holds\nthe questions\ni’m finally ready to ask.\n.\nand maybe that’s what depth is.\nnot something you dig into,\nbut something you grow beside.\nlike how silence can be full.\nor how a moment can stretch\nand still feel complete.\n.\nin these quiet corners,\ni feel like the world stops measuring me.\nnot by how much i do,\nbut by how much i notice.\nhow long i can sit with a thought\nwithout running away.\n.\ntruth doesn’t always knock.\nsometimes, it just\nwaits.\n.\nand you find it\nin a dusty bookstore,\nin the gold frame of a stranger’s portrait,\nin the way a page turns slowly—\nnot revealing,\nbut remembering.'}, {'poem_id': 'P06', 'title': 'and the canary kept singing', 'poem_text': 'what if the spear missed?\n\nwhat if the gods blinked —\njust once —\nand the little bird slipped through their fingers?\n\nrue lives.\n\nnot untouched.\nnot unharmed.\nbut changed.\n\nsomething in her eyes turns glassy,\nlike she’s seen past the veil\nand returned with silence stitched beneath her tongue.\n\nkatniss finds her in the hollow of a tree,\nblood dried like rust along her arms,\nstill singing —\na lullaby with no melody,\njust breath and tremble and grief.\n\nthey move through the arena like ghosts.\nnot hunted.\nhaunting.\n\nrue speaks less now.\nonly in riddles.\nonly in dreams.\n\n“they built this place on bones,” she whispers,\n“so it will starve when we give it none.”\n\nthey starve the game.\nrefuse violence.\nrefuse story.\n\nthey smear mud on the cameras,\nstring up dolls from branches,\ntear the eyes from the trees.\n\npanem watches.\nthe capitol seethes.\n\nthese are not girls.\nthey are omens.\n\nand when the final order comes —\n“one must die” —\nthey do not flinch.\n\nrue walks to the center.\nbarefoot.\nno weapon.\nno tears.\n\n“you want blood?”\nher voice cracks like a hymn.\n“then choke on mine.”\n\nshe opens her arms.\ncalls down fire.\n\nthey call it a system failure.\na broadcast glitch.\na technical tragedy.\n\nbut the districts know.\n\nthey saw the girl who lived\nwalk into the flames\nwith a smile that didn’t belong in this world.\n\nkatniss is crowned.\nbut her eyes are empty.\nshe dreams of twisted roots,\na songbird without a beak,\nand smoke that tastes like rue’s breath.\n\nthe games ended.\nbut something older woke up that day.\nand it sings still —\nlow, cold, and vengeful\nin every crack of the stone.'}, {'poem_id': 'P07', 'title': 'the orchard fed her bones', 'poem_text': '.\nthey said the girl died clean.\n.\nthey lied.\n.\nrue did not die.\nrue was taken.\nswallowed by the trees,\nstitched back together with rot and root,\nreassembled in the dark belly of the arena —\na girl reborn, not in grace, but in vengeance.\n.\nwhen katniss finds her,\nshe is not whole.\nher eyes are too black,\ntoo still.\nher voice, when it comes, sounds like something buried too long.\n.\n"they tried to harvest me,” she says,\n“but the orchard took me first.”\n.\n.\n.\nshe bleeds sap.\nshe sings in tongues.\nshe moves like something that remembers being human\nbut isn’t anymore.\n.\nkatniss follows her anyway.\nof course she does.\neven when the air turns sour,\neven when the trees whisper names not spoken in decades,\neven when the ground breathes beneath their feet.\n.\nthe arena begins to decay.\n.\nmoss spreads like rot across the cornucopia.\ncameras crack.\nmutts gnaw at their own paws.\nthe sky weeps black rain.\n.\nand rue moves through it all, untouched,\na child-shaped plague.\n.\nthe Capitol sends in fire.\n.\nrue laughs.\nit is the sound of teeth on bone.\n.\n“they want a show?”\nher smile is a wound.\n“then let them choke on ruin.”\n.\n.\n.\nshe does not die.\nshe unravels.\n.\nat the final countdown, when only two remain,\nshe walks to the center of the arena\nand opens her mouth wide.\n.\nfrom her throat pours ash,\ncarrion birds,\nfragments of hymns older than the Capitol itself.\n.\nthe screen goes black.\nno victor.\nno anthem.\n.\njust silence.\n.\nin the weeks that follow,\ndistrict children begin waking with soil under their nails.\nthe woods grow wrong.\nmockingjays fall from the sky mid-song.\nand in the corners of mirrors,\nthey see her —\nsmall, barefoot, watching.\n.\nstill humming.'}, {'poem_id': 'P08', 'title': 'the phoenix knows', 'poem_text': "the phoenix knows\nthat burning isn't always violent.\nsometimes, it's quiet—\na slow undoing,\na surrender to gravity\nbefore the sky remembers your name.\n.\nshe doesn’t rise for glory.\nshe rises because there’s nothing left\nbut the choice\nto begin again.\n.\nash is not the end.\nit is memory—\nof what was heavy,\nwhat had to fall apart.\n.\nand from it,\nshe gathers herself—\nnot new,\nbut changed.\n.\nnot untouched,\nbut honest.\n.\neach feather a story\nof what the fire took,\nand what it couldn’t.\n.\nthe phoenix never asks\nto be seen.\nshe only asks\nto survive.\nand still,\nshe rises."}, {'poem_id': 'P09', 'title': 'wax child', 'poem_text': 'i saw her hold him like he was whole —\na mother and son, a stitched-up soul.\nthey walked through noise,\nbut she only heard\nhis laugh,\nhis little word.\n\nthen —\na scream in steel.\na shot, too real.\nher leg caught fire\nbut her voice stayed calm.\nshe only ached\nfor her baby’s palm.\n\nhe stood so still,\nin the heart of it all,\nwatched her stumble,\nwatched her fall.\na tear slipped down —\nand he began to fade.\nhis skin ran soft.\nhis shape unmade.\n\nnot a boy.\nnever one.\njust wax beneath a dying sun.\nand still,\nshe reached.\nshe called his name\nlike love alone\ncould stop the flame.\n\nhe broke.\nlimbs cracked,\ncold and light.\nand she held him close\nlike it was all right.\n\nthey said\n“he was never real.”\nbut what do they know\nof the way we feel?\n\nshe rocked him slow\nwith empty hands,\nstill humming songs\nhe’d never understand.\n\nand when they pulled\nher grief away,\nher arms were shaped\nfor him to stay.'}, {'poem_id': 'P10', 'title': 'It Starts With Thought', 'poem_text': "Before the hand can draw a line,\nbefore the stars are made to shine,\nbefore the world can call it mine—\na thought must whisper, soft, divine.\n\nIt doesn't shout, it doesn’t show,\nit plants its seeds and lets them grow.\nWhat blooms in time, the world will know—\nbut roots run deep where thoughts first go.\n\nA moment’s pause, a silent spark,\ncan shift the day, can light the dark.\nThe smallest drift, the slightest mark—\ncan leave behind a lasting arc.\n\nSo guard your mind, both wild and wide,\nfor that’s the place where truths decide.\nAll that you do, and what you hide—\nbegan with thought, not step or stride."}, {'poem_id': 'P11', 'title': 'The Knowing Quiet', 'poem_text': "Listen to the wind—it speaks in sighs,\nin dancing leaves and open skies.\nIt tells of things you used to feel,\nbefore the world taught you to kneel.\n\nListen to the silence—it holds space,\nfor every tear you didn't face.\nIt doesn't judge, it doesn't flee,\nit simply waits for you to see.\n\nListen to your heart—it knows the way,\nthrough broken nights and heavy days.\nIt may be soft, but it is strong—\nit hums the truth when all feels wrong.\n\nThere is a voice beneath the noise,\nbeyond the crowds, beyond the choice.\nIt lives in stillness, breath, and flame—\nand when you hear it, you're not the same."}, {'poem_id': 'P12', 'title': 'Call of the Brave', 'poem_text': 'There was never a promise of safety,\nonly the whisper of stars beyond sight—\na voice in the wind that said go,\nthat dared the heart to ignite.\n\nThe path was not paved with answers,\nnor lit by lanterns of ease;\nit twisted through thunder and silence,\nthrough forests that brought you to knees.\n\nBut those who walked it with wonder,\nwith fire in marrow and bone,\nfound treasures not made of silver—\nbut courage, carved deep in their own.\n\nSo cast off the cloak of comfort,\nstep past the walls of known.\nFor life is not meant to be measured—\nbut tasted, and risked, and grown.'}])
tests=pd.DataFrame([{'poem_id': 'P02', 'original': 'these mountains that you are holding,\nyou were only supposed to climb.', 'meaning': 'Challenges are obstacles to overcome rather than burdens to carry indefinitely.', 'literal': 'These mountains are difficult to climb while holding heavy equipment.', 'meaning_alt': 'Stop carrying the weight of your struggles and work through them instead.', 'meaning_overlap': 'The mountains you carry are challenges to climb, not lifelong burdens.', 'literal_overlap': 'Holding a mountain climbing permit allows you to climb the mountain.', 'literal_low': 'Hikers transport equipment uphill.'}, {'poem_id': 'P01', 'original': 'faces are only geometry.\nlight resting temporarily on bone.', 'meaning': 'Physical appearance alone cannot capture inner experience.', 'literal': 'Facial geometry and bone structure determine how light falls across a portrait.', 'meaning_alt': 'A person’s visible features reveal little about their emotional life.', 'meaning_overlap': 'The geometry of faces and light on bone cannot reveal inner feelings.', 'literal_overlap': 'Geometry describes faces and light reveals bone contours.', 'literal_low': 'Portrait artists study anatomy and illumination.'}, {'poem_id': 'P04', 'original': 'broken glass reflects more light\nthan something whole ever could.', 'meaning': 'Hardship can lead to greater strength and complexity.', 'literal': 'Broken glass reflects light at different angles from a whole glass panel.', 'meaning_alt': 'Painful experiences can create new forms of beauty and resilience.', 'meaning_overlap': 'Like broken glass reflecting light, a wounded person can discover new strength.', 'literal_overlap': 'Broken glass reflects light from more surfaces than whole glass.', 'literal_low': 'Fragments of a mirror scatter illumination.'}, {'poem_id': 'P03', 'original': 'the river doesn’t rush all the time—\nsometimes it drifts,\nsometimes it rests,\nbut it always flows.\n\nso do you.', 'meaning': 'Personal growth continues even during periods of apparent inactivity.', 'literal': 'A river sometimes flows rapidly and sometimes drifts slowly depending on its surroundings.', 'meaning_alt': 'Moving slowly or resting does not mean you have stopped developing.', 'meaning_overlap': 'Like a river that drifts and flows, you can grow even when you rest.', 'literal_overlap': 'A river drifts slowly or rushes as its flow changes.', 'literal_low': 'Watercourses vary in speed with terrain.'}, {'poem_id': 'P08', 'original': 'ash is not the end.\nit is memory—', 'meaning': 'The aftermath of hardship retains traces of experiences that transformed us.', 'literal': 'Ash is the residue left after material burns.', 'meaning_alt': 'What remains after adversity carries traces of the past.', 'meaning_overlap': 'Ash is the memory of what we lost while surviving hardship.', 'literal_overlap': 'Ash is what remains at the end of combustion.', 'literal_low': 'Burning combustible matter leaves solid residue.'}, {'poem_id': 'P09', 'original': 'and when they pulled\nher grief away,\nher arms were shaped\nfor him to stay.', 'meaning': 'Absence does not erase the habits and attachments formed by love.', 'literal': 'Her arms retained their shape after the object was pulled away.', 'meaning_alt': 'Grief persists in the habits of someone who has lost a loved one.', 'meaning_overlap': 'Her arms still hold the shape of love even after he is pulled away.', 'literal_overlap': 'Her arms stay in the same shape after the object is pulled away.', 'literal_low': 'A person maintains a posture after an item is removed.'}])
tests.to_csv(out/"challenge_prompts.csv",index=False)
print(tests[["poem_id","original"]])

## Encode excerpts, complete poems and six candidate paraphrases
Use normalized embeddings with token-weighted chunk pooling for long inputs. Verify re-tokenized chunks fit the model limit. TF-IDF is fitted once on the evaluation texts; it is a descriptive lexical baseline.

In [ ]:
cols=['meaning','meaning_alt','meaning_overlap','literal','literal_overlap','literal_low']
texts=list(dict.fromkeys(poems.poem_text.tolist()+tests.original.tolist()+[s for col in cols for s in tests[col]]))
models={'MiniLM':'sentence-transformers/all-MiniLM-L6-v2','MPNet':'sentence-transformers/all-mpnet-base-v2','BGE':'BAAI/bge-base-en-v1.5'}
lookup={s:i for i,s in enumerate(texts)}
matrices={}
metadata={}
def encode(text,m):
    tok=m.tokenizer
    ids=tok.encode(text,add_special_tokens=False,truncation=False,verbose=False)
    limit=m.max_seq_length-tok.num_special_tokens_to_add(pair=False)
    if len(ids)<=limit:
        return m.encode(text,normalize_embeddings=True)
    chunks=[]; weights=[]; start=0
    while start<len(ids):
        end=min(start+limit,len(ids))
        while True:
            s=tok.decode(ids[start:end],skip_special_tokens=True)
            if len(tok.encode(s,add_special_tokens=True,truncation=False,verbose=False))<=m.max_seq_length: break
            end-=1
            assert end>start
        chunks.append(s); weights.append(end-start); start=end
    v=np.average(m.encode(chunks,normalize_embeddings=True),axis=0,weights=weights)
    return v/max(np.linalg.norm(v),1e-12)
for name,mid in models.items():
    print('Loading',name)
    m=SentenceTransformer(mid,device='cuda' if torch.cuda.is_available() else 'cpu')
    metadata[name]={'id':mid,'max_seq_length':m.max_seq_length}
    vectors=np.vstack([encode(s,m) for s in texts])
    matrices[name]=cosine_similarity(vectors)
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
matrices['TF-IDF']=cosine_similarity(TfidfVectorizer(ngram_range=(1,1)).fit_transform(texts))
(out/'metadata.json').write_text(json.dumps(metadata,indent=2))


In [ ]:
def overlap(a,b):
    a=set(re.findall(r"\b\w+\b",a.lower())); b=set(re.findall(r"\b\w+\b",b.lower()))
    return len(a&b)/max(len(a|b),1)
records=[]
for r in tests.to_dict('records'):
    full=poems.set_index('poem_id').loc[r['poem_id'],'poem_text']
    for context,anchor in [('excerpt',r['original']),('full_poem',full)]:
        for name,mat in matrices.items():
            for col in cols:
                records.append(dict(poem_id=r['poem_id'],context=context,model=name,candidate=col,kind='figurative' if col.startswith('meaning') else 'literal',similarity=float(mat[lookup[anchor],lookup[r[col]]]),lexical_jaccard=overlap(anchor,r[col])))
scores=pd.DataFrame(records); scores.to_csv(out/'all_scores.csv',index=False)
print(scores.head())


## Compare margins
Positive margin means the figurative candidate scored higher. Compare original candidates, alternative low-overlap candidates and intended high-overlap candidates separately. These are hand-crafted contrasts; measured lexical overlap is exported so we can inspect imperfect matching.

In [ ]:
contrasts={'original':('meaning','literal'),'alternate_low':('meaning_alt','literal_low'),'overlap_matched':('meaning_overlap','literal_overlap')}
wide=scores.pivot(index=['poem_id','context','model'],columns='candidate',values='similarity').reset_index()
margins=pd.concat([wide[['poem_id','context','model']].assign(contrast=k,margin=wide[a]-wide[b]) for k,(a,b) in contrasts.items()],ignore_index=True)
margins['figurative_wins']=margins.margin>0
margins.to_csv(out/'margins.csv',index=False)
summary=margins.groupby(['model','context','contrast']).agg(mean_margin=('margin','mean'),wins=('figurative_wins','sum'),n=('margin','size')).reset_index()
summary.to_csv(out/'summary.csv',index=False)
display(summary)
context=margins.pivot(index=['poem_id','model','contrast'],columns='context',values='margin').reset_index()
context['context_gain']=context.full_poem-context.excerpt
context.to_csv(out/'context_gain.csv',index=False)
for contrast in contrasts:
    sub=summary[summary.contrast==contrast].pivot(index='model',columns='context',values='mean_margin')
    ax=sub.plot.bar(figsize=(9,5),rot=0); ax.axhline(0,color='black',linewidth=1); ax.set_ylabel('Mean figurative minus literal cosine'); ax.set_title(contrast); plt.tight_layout(); plt.savefig(out/f'{contrast}.png',dpi=200); plt.show()


## Export results
Candidate prompts contain excerpts of private poetry. Review before sharing publicly. No significance claims: six excerpts are not an independent representative sample, and model scores are not comprehension scores.

In [ ]:
shutil.make_archive('poetry_followup_results','zip',out)
files.download('poetry_followup_results.zip')